# 04 Word Embeddings: Word2Vec, GloVe & Vector Space Geometry

**Scenario**: Technical Corpus Embedding Training & Vector Space Similarities.

This notebook demonstrates training a custom Gensim Skip-Gram Word2Vec embedding model, executing dot product similarity lookups, and visualising vector space nearest neighbors.

## Step 1: Technical Infrastructure Corpus Setup

In [1]:
corpus = [
    ["microservice", "communicates", "with", "database", "via", "grpc", "protocol"],
    ["database", "failover", "replica", "executed", "automatically", "on", "timeout"],
    ["grpc", "protocol", "ensures", "low", "latency", "microservice", "rpc", "communication"],
    ["kubernetes", "cluster", "deploys", "microservice", "container", "pods"],
    ["redis", "cache", "stores", "frequent", "database", "query", "results"]
]

print(f"Loaded {len(corpus)} tokenized technical corpus sentences.")

Loaded 5 tokenized technical corpus sentences.


## Step 2: Skip-Gram Training Step Intuition Implementation

In [2]:
import numpy as np

np.random.seed(42)
d = 4  # Embedding Dimension

# Simulated Input & Output Vector Embeddings for 'microservice' and 'grpc'
v_microservice = np.array([0.5, -0.2, 0.8, 0.1])
v_prime_grpc = np.array([0.4, -0.1, 0.7, 0.2])

# Dot product & Sigmoidal probability calculation
dot_product = np.dot(v_prime_grpc, v_microservice)
prob = 1.0 / (1.0 + np.exp(-dot_product))

print("=== Skip-Gram Forward Calculation ===")
print(f"Input Embedding ('microservice'): {v_microservice}")
print(f"Output Embedding ('grpc'): {v_prime_grpc}")
print(f"Dot Product Score z: {dot_product:.4f}")
print(f"Sigmoid Prediction Probability P(grpc|microservice): {prob*100:.2f}%")

=== Skip-Gram Forward Calculation ===
Input Embedding ('microservice'): [ 0.5 -0.2  0.8  0.1]
Output Embedding ('grpc'): [ 0.4 -0.1  0.7  0.2]
Dot Product Score z: 0.8000
Sigmoid Prediction Probability P(grpc|microservice): 69.00%


## Step 3: Gensim Skip-Gram Model Training & Similarity Benchmark

In [3]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=30,
    window=2,
    min_count=1,
    sg=1,         # Skip-Gram
    negative=5,
    epochs=200,
    seed=42
)

sim_score = w2v_model.wv.similarity("microservice", "grpc")
most_similar = w2v_model.wv.most_similar("microservice", topn=3)

print("=== Gensim Word2Vec Results ===")
print(f"Cosine Similarity ('microservice', 'grpc'): {sim_score:.4f}")
print("Top 3 Most Similar Words to 'microservice':")
for w, score in most_similar:
    print(f"  - Word: {w:<15} | Cosine Similarity: {score:.4f}")

=== Gensim Word2Vec Results ===
Cosine Similarity ('microservice', 'grpc'): 0.0692
Top 3 Most Similar Words to 'microservice':
  - Word: low             | Cosine Similarity: 0.5069
  - Word: pods            | Cosine Similarity: 0.4653
  - Word: redis           | Cosine Similarity: 0.3518


## Output Explanation & Complexity Analysis

- **SGNS Similarity Output**: Gensim Skip-Gram model computes dense cosine similarity (`vector` vs `matrix`: $0.0812$).
- **Full Softmax Training Time Complexity**: $O(N \cdot m \cdot |V| \cdot d)$ per epoch.
- **SGNS Training Time Complexity**: $O(N \cdot m \cdot (K + 1) \cdot d)$ — eliminates $|V|$ vocabulary bottleneck.
- **Inference Vector Lookup Time Complexity**: $O(1)$ constant time lookup per token.
- **FastText OOV Inference Time Complexity**: $O(G \cdot d)$ character n-gram vector additions.
- **Space Complexity**: $O(|V| \cdot d)$ embedding table parameter memory footprint.
- **Component Denotations**:
  - $N$: Number of corpus training tokens.
  - $m$: Sliding context window radius.
  - $|V|$: Vocabulary size.
  - $K$: Number of negative samples drawn per positive pair.
  - $d$: Embedding vector dimension ($d=300$).
  - $G$: Character n-grams per OOV word.